# Working with the Labour Force Survey (in Python)
## UK Data Service datasets

In this notebook, we aim to write functionality to simplify locating, looking up, renaming and recoding variables in UKDS datasets.

UKDS data dictionaries in the **Understanding Society** and **Labour Force Survey** datasets are buried in `rtf` files in nested archives. Unzipping these manually can be effort intensive. Recoding variables even more so. We can Python and R to automate parts of these processes. (1) We can write functions that unzip, relocate and rename files should we wish to simplify access to individual datasets. (2) We can write functions that extract the data dictionary information and store them in useful dictionaries we can iterate through when we need to look up or recode a variable.

In [1]:
# Libraries
import numpy as np
#import pandas as pd
import polars as pl

import re
import io
import os

from da_helpers import *

In [2]:
# LFS 2024-2026 archives folder
data_root = r"F:\OneDrive\coding_files\data\lfs_2024-2026"

print("Did we get the path right?")
os.path.exists(data_root)

Did we get the path right?


True

## Archives

UKDS datasets come in zip files. We list the required files first.

In [3]:
# List all tab archives
lfs_files = [f for f in crawl_folder(data_root) if f.endswith("zip") and "tab_" in f]

print("Available tab-delimited datasets:")
lfs_files

Available tab-delimited datasets:


['F:\\OneDrive\\coding_files\\data\\lfs_2024-2026\\9264tab_E4EB2F2F5DC05C27D6DC9DA4527F4E41B1CE7090AD7E6C35FCBB0F9EBC11D35D_V1.zip',
 'F:\\OneDrive\\coding_files\\data\\lfs_2024-2026\\9270tab_B4A81ED99DAEB16984A7A74E1EB3D3E043C0ECB23E739F1030C1B9F2D3509FDD_V1.zip',
 'F:\\OneDrive\\coding_files\\data\\lfs_2024-2026\\9271tab_9CF3D2FC4DC44C17A8FC36FC16E80665224ECCB03C396FDBCE2EA11F6986241D_V1.zip',
 'F:\\OneDrive\\coding_files\\data\\lfs_2024-2026\\9272tab_1EAF2F77A3F94166741B7AF117053DA3DB1D3B1D019C6CC6FA4062C54E92BCE4_V1.zip',
 'F:\\OneDrive\\coding_files\\data\\lfs_2024-2026\\9297tab_D42E65C2B512DF771A004B37D76F4E31F591A47B63069B0CBF199A8A6CCC350A_V1.zip',
 'F:\\OneDrive\\coding_files\\data\\lfs_2024-2026\\9303tab_0AA54F15FACE9935D2A646A6E0B6862CF9D955C13B92A8326CC4ABA9463BFDF2_V1.zip',
 'F:\\OneDrive\\coding_files\\data\\lfs_2024-2026\\9323tab_E0073C655655038B6C7002C5BB0B9EA162FCE2D25CB1DE1EB083372246D5B2CB_V1.zip',
 'F:\\OneDrive\\coding_files\\data\\lfs_2024-2026\\9328tab_9A366D1B44

We test access to the archived files on the first archive in the list. Note that the block is loop-ready. We can wrap it in a `for` loop and iterate through the entire list.

In [4]:
# Itemise the contents of each archive
i = 0

zip_inventory = {}
for zip_path in lfs_files:
    zip_name = re.match(r"\d+", os.path.basename(zip_path)).group()
    zip_inventory[zip_name] = itemise_archive(lfs_files[i])

print("Contents of an archive:")
zip_inventory

Contents of an archive:


{'9264': ['UKDA-9264-tab/9264_file_information.rtf',
  'UKDA-9264-tab/mrdoc/ukda_data_dictionaries.zip',
  'UKDA-9264-tab/mrdoc/pdf/biennial_lfs_aps_reweighting_policy.pdf',
  'UKDA-9264-tab/mrdoc/pdf/coronavirus_and_its_impact_on_the_labour_force_survey.pdf',
  'UKDA-9264-tab/mrdoc/pdf/labour_force_survey_newsletter_april_2020.pdf',
  'UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol10_analysis_of_data_collected_by_lfs_2024.pdf',
  'UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol11_longitudinalguide2022.pdf',
  'UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol1_background_2024.pdf',
  'UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol2_questionnaire_2022jscombined.pdf',
  'UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol3_variabledetails2022js.pdf',
  'UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol3_variable_details_1992-2002.pdf',
  'UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol4a_educationsupplement2022.pdf',
  'UKDA-9264-tab/mrdoc/pdf/lfs_user_guide_vol4_derived_variables_2020.pdf',
  'UKDA-9264-tab/mrdoc/pdf/lfs_user

## Extracting files

We can extract a file and move it to a more convenient location under a more convenient name.

In [5]:
src = r"F:\temp\src\tst.txt"
tgt = r"F:\temp\tgt"

def copy_file(source_file_path, target_folder_path):
    """
    Copies the file on `source_file_path` into `target_folder_path` folder.
    input:              source_file_path: str, target_file_path: str
    output:             None
    """
    import os
    import shutil

    if not os.path.exists(source_file_path):
        raise FileNotFoundError("The source file does not exist.")

    file_name = os.path.basename(source_file_path)
    target_file_path = os.path.join(target_folder_path, file_name)
    os.makedirs(target_folder_path, exist_ok=True)

    shutil.copy(source_file_path, target_file_path)
    print(f"'{file_name}' copied to '{target_folder_path}'.")

copy_file(src, tgt)

'tst.txt' copied to 'F:\temp\tgt'.


In [6]:
i = 0
archive_path = lfs_files[i]
archived_file_path = [f for f in itemise_archive(archive_path) if "file_information" in f][0]
tgt_folder = r"F:\temp\tgt"

def copy_archived_file(archive_path, archived_file_path, target_folder_path):
    """
    Extracts the file on `archived_file_path` into `target_folder_path` folder.
    input:              archive_path: str, archived_file_path: str, target_folder_path: str
    output:             None
    """
    import os
    import shutil

    if not os.path.exists(archive_path):
        raise FileNotFoundError("The archive file does not exist.")

    file_name = os.path.basename(archived_file_path)
    target_file_path = os.path.join(target_folder_path, file_name)
    os.makedirs(target_folder_path, exist_ok=True)

    with extract_file(archive_path, archived_file_path) as src, open(target_file_path, "wb") as out:
        shutil.copyfileobj(src, out)
    print(f"'{file_name}' extracted into '{target_folder_path}'.")

copy_archived_file(archive_path, archived_file_path, tgt_folder)

'9264_file_information.rtf' extracted into 'F:\temp\tgt'.


## Data dictionaries

Dataset specific documentation is stored in `rtf` files in nested archives. We need to read up the `rtf` file two layers deep. We get a dictionary exactly as we see it when we open the `rtf` file in MS Word.

In [7]:
def read_data_dictionary(zip_file):
    """
    Reads up an rtf data dictionary from a nested zip file.
    libraries:          import io
    custom functions:   extract_file, itemise_archive, read_rtf
    input:              zip_file: str
    output:             str
    """
    nested_zip_path = [f for f in itemise_archive(zip_file) if f.endswith("zip") and "data_dictionaries" in f][0]

    outer_zip = io.BytesIO(
            extract_file(
                zip_file,
                nested_zip_path
            ).read()
        )

    text = read_rtf(
            extract_file(
                outer_zip,
                [f for f in itemise_archive(outer_zip) if f.endswith("rtf")][0]
            )
        )
    return text

# Extract a data dictionary
i = 0

data_dictionary_doc = read_data_dictionary(lfs_files[i])

print(data_dictionary_doc)


UK Data Archive Data Dictionary

File-level information:

File Name = 		lfsp_jm24_eul_pwt24
Number of variables = 	801
Number of cases = 	50783


Variable-level information:

Pos. = 1	Variable = ACCDNT	Variable label = Accident resulting in injury at work
This variable is  numeric, the SPSS measurement level is NOMINAL
SPSS user missing values = -8.0 thru -9.0
	Value label information for ACCDNT
	Value = -9.0	Label = Does not apply
	Value = -8.0	Label = No answer
	Value = 1.0	Label = Yes
	Value = 2.0	Label = No

Pos. = 2	Variable = ACCKIND	Variable label = Type of Accident
This variable is  numeric, the SPSS measurement level is NOMINAL
SPSS user missing values = -8.0 thru -9.0
	Value label information for ACCKIND
	Value = -9.0	Label = Does not apply
	Value = -8.0	Label = No answer
	Value = 1.0	Label = Contact with moving machinery
	Value = 2.0	Label = Hit by moving, flying or falling object
	Value = 3.0	Label = Hit by a moving vehicle
	Value = 4.0	Label = Hit something fixed or stati

We are actually interested in automating selection, renaming and access. A dicitonary mapping variable names onto the respective column position and variable label is therefore a useful idea.

In [8]:
def variable_dictionary(data_dictionary_doc):
    """
    Returns a dictionary of {variable: (position, label)} from a data dictionary.
    libraries:          import re
    input:              data_dictionary_doc: str
    output:             dict
    """
    var_pat = re.compile(r"^Pos\. = (\d+)\tVariable = (\w+)\tVariable label = (.*)$", re.M)

    var_dict = {
        re.sub(r"(\d+)$", r"_\1", var.lower()): (pos, lab) 
        for (pos, var, lab) 
        in var_pat.findall(data_dictionary_doc)
    }
    return var_dict

# Map variables onto (col position, description) pairs
variable_dictionary(data_dictionary_doc)

{'accdnt': ('1', 'Accident resulting in injury at work'),
 'acckind': ('2', 'Type of Accident'),
 'accurh_1': ('3', 'injuries'),
 'accurh_2': ('4', 'injuries'),
 'accurh_3': ('5', 'injuries'),
 'accurh_4': ('6', 'injuries'),
 'acthr': ('7', 'Actual hours excluding overtime'),
 'acthr_2': ('8', '2nd job actual hours including overtime'),
 'actpot': ('9', 'Actual hours of paid overtime'),
 'actuot': ('10', 'Actual hours of unpaid overtime'),
 'actwkdy_1': ('11', 'Days scheduled to work on - 1'),
 'actwkdy_2': ('12', 'Days scheduled to work on - 2'),
 'actwkdy_3': ('13', 'Days scheduled to work on - 3'),
 'actwkdy_4': ('14', 'Days scheduled to work on - 4'),
 'actwkdy_5': ('15', 'Days scheduled to work on - 5'),
 'actwkdy_6': ('16', 'Days scheduled to work on - 6'),
 'actwkdy_7': ('17', 'Days scheduled to work on - 7'),
 'addjob': ('18', 'Looking for new or additional job'),
 'advhst': ('19', 'Number of advanced highers'),
 'age': ('20', 'Age of respondent'),
 'ageeul': ('21', 'Age bands'

It is also a good idea to construct another dictionary of dictionaries mapping variable names onto codes and values. This will come handy when we want to search an existing schema or recode it.

In [9]:
def code_dictionary(data_dictionary_doc):
    """
    Returns a dictionary of {variable: {code: value}} from a data dictionary.
    libraries:          import re
    input:              data_dictionary_doc: str
    output:             dict
    """
    kv_pat = re.compile(r"^Pos\. = \d+\tVariable = (\w+)\t.*$|^\tValue = (\S+)\s*\tLabel = (.*)$", re.M)

    code_dict = {}
    for var, code, label in kv_pat.findall(data_dictionary_doc):
        if var:
            k = re.sub(r"(\d+)$", r"_\1", var.lower())
            code_dict[k] = {}
        else:
            code_dict[k][code] = label
    return code_dict

# Map variables onto {code: label} sets
code_dictionary(data_dictionary_doc)

{'accdnt': {'-9.0': 'Does not apply',
  '-8.0': 'No answer',
  '1.0': 'Yes',
  '2.0': 'No'},
 'acckind': {'-9.0': 'Does not apply',
  '-8.0': 'No answer',
  '1.0': 'Contact with moving machinery',
  '2.0': 'Hit by moving, flying or falling object',
  '3.0': 'Hit by a moving vehicle',
  '4.0': 'Hit something fixed or stationary',
  '5.0': 'Injured handling, lifting or carrying',
  '6.0': 'Slipped, tripped or fell on same level',
  '7.0': 'Fell from a height',
  '8.0': 'Trapped by sthing collapsing overtng',
  '9.0': 'Drowned or asphyxiated',
  '10.0': 'Expd to, or in contact w., harmful subs',
  '11.0': 'Exposed to fire',
  '12.0': 'Exposed to an explosion',
  '13.0': 'Contact w. electy or elec. dischge',
  '14.0': 'Injured by an animal',
  '15.0': 'Physically assaulted by a person',
  '16.0': 'Another kind of accident'},
 'accurh_1': {'-9.0': 'Does not apply',
  '-8.0': 'No answer',
  '1.0': 'lose consciousness',
  '2.0': 'hypothermiaheat',
  '3.0': 'resuscitation',
  '4.0': 'hospital 

## Datasets

Let's test access to an archived dataset.

In [10]:
# Test df extract
i = 0
tab_file_name = [f for f in itemise_archive(lfs_files[i]) if f.endswith("tab")][0]

text = extract_file(lfs_files[i], tab_file_name)
print(read_tab_pl(text, nrows=5, cols=list(range(5))))

shape: (5, 5)
┌────────┬─────────┬──────────┬──────────┬──────────┐
│ accdnt ┆ acckind ┆ accurh_1 ┆ accurh_2 ┆ accurh_3 │
│ ---    ┆ ---     ┆ ---      ┆ ---      ┆ ---      │
│ i64    ┆ i64     ┆ i64      ┆ i64      ┆ i64      │
╞════════╪═════════╪══════════╪══════════╪══════════╡
│ -9     ┆ -9      ┆ -9       ┆ -9       ┆ -9       │
│ -9     ┆ -9      ┆ -9       ┆ -9       ┆ -9       │
│ -9     ┆ -9      ┆ -9       ┆ -9       ┆ -9       │
│ 2      ┆ -9      ┆ -9       ┆ -9       ┆ -9       │
│ -9     ┆ -9      ┆ -9       ┆ -9       ┆ -9       │
└────────┴─────────┴──────────┴──────────┴──────────┘
